## Review from Data Exploration in 01_exploration.ipynb:

In [54]:
import os
import glob
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)

In [55]:
file_list = glob.glob(r"C:\Users\zahir\CRMLSSold*.csv")
dataframes = [pd.read_csv(file) for file in file_list]
df = pd.concat(dataframes, ignore_index=True)

C:\Users\zahir\AppData\Local\Temp\ipykernel_36624\405482455.py:2: DtypeWarning: Columns (78,79) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file) for file in file_list]
C:\Users\zahir\AppData\Local\Temp\ipykernel_36624\405482455.py:2: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file) for file in file_list]
C:\Users\zahir\AppData\Local\Temp\ipykernel_36624\405482455.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file) for file in file_list]
C:\Users\zahir\AppData\Local\Temp\ipykernel_36624\405482455.py:2: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(file) for file in file_list]


# Missing Data Handling

In [56]:
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")]
missing_percentages = df.isna().sum()[df.isna().sum() > 0] / len(df)
missing_percentages

Flooring                        0.357766
ViewYN                          0.099001
WaterfrontYN                    0.999562
BasementYN                      0.975581
PoolPrivateYN                   0.101582
                                  ...   
MiddleOrJuniorSchoolDistrict    1.000000
latfilled                       0.577074
lonfilled                       0.577074
BuyerAgentAOR                   0.312165
ListAgentAOR                    0.303031
Length: 69, dtype: float64

In [57]:
ZERO_INFO_COLS = [
    "TaxAnnualAmount", "AboveGradeFinishedArea", "TaxYear",
    "ElementarySchoolDistrict", "CoveredSpaces", "BusinessType",
    "MiddleOrJuniorSchoolDistrict", "FireplacesTotal",
]

ID_COLS = [
    "ListingKey", "ListingKeyNumeric", "ListingId",
    "UnparsedAddress", "StreetNumberNumeric",
]

AGENT_OFFICE_COLS = [
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "ListAgentEmail", "ListAgentAOR",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId", "BuyerAgentAOR",
    "CoListAgentFirstName", "CoListAgentLastName", "CoBuyerAgentFirstName",
    "ListOfficeName", "BuyerOfficeName", "CoListOfficeName", "BuyerOfficeAOR",
    "BuyerAgencyCompensation", "BuyerAgencyCompensationType",
]

LISTING_ARTIFACT_COLS = ["ListPrice", "OriginalListPrice", "DaysOnMarket"]

TARGET = "ClosePrice"

DROP_NOW = (ZERO_INFO_COLS + ID_COLS + AGENT_OFFICE_COLS
            + LISTING_ARTIFACT_COLS)

before_cols = df.shape[1]
df = df.drop(columns=[c for c in DROP_NOW if c in df.columns])
print(f"Dropped {before_cols - df.shape[1]} columns, {df.shape[1]} remain")

Dropped 34 columns, 48 remain


In [58]:
YN_COLS = ["ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN",
           "AttachedGarageYN", "NewConstructionYN", "FireplaceYN"]

for c in YN_COLS:
    print(f"\n{c}:")
    print(df.groupby(df[c].astype("object"), dropna=False)[TARGET].agg(["mean", "count"]))


ViewYN:
                mean   count
ViewYN                      
False   1.178986e+06  142959
True    1.372029e+06  216679
NaN     1.192262e+06   39517

WaterfrontYN:
                      mean   count
WaterfrontYN                      
NaN           1.284269e+06  398980
True          3.161635e+06     175

BasementYN:
                    mean   count
BasementYN                      
NaN         1.266897e+06  389408
True        2.012028e+06    9747

PoolPrivateYN:
                       mean   count
PoolPrivateYN                      
False          1.084415e+06  300866
True           1.664933e+06   57742
NaN            2.233236e+06   40547

AttachedGarageYN:
                          mean   count
AttachedGarageYN                      
False             1.251821e+06   59422
True              1.280212e+06  293624
NaN               1.359050e+06   46109

NewConstructionYN:
                           mean   count
NewConstructionYN                      
NaN                1.411621e+06   39

In [60]:
df.dropna(subset=['PoolPrivateYN', 'AttachedGarageYN', 'NewConstructionYN', 'FireplaceYN', 'ViewYN' ])
df[['WaterfrontYN', 'BasementYN']] = df[['WaterfrontYN', 'BasementYN']].fillna(False)

# Train / Test Splitting

In [66]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
last_date = df["CloseDate"].max()
last_month_end = last_date.to_period("M").end_time

df["CloseMonth"] = df["CloseDate"].dt.to_period("M")
all_months = sorted(df["CloseMonth"].unique())

TEST_MONTH = all_months[-1]
print(f"TEST_MONTH = {TEST_MONTH}")
print("\nRows per month (most recent 6): \n")
print(df["CloseMonth"].value_counts().sort_index().tail(6))

TEST_MONTH = 2026-05

Rows per month (most recent 6): 

CloseMonth
2025-12    10455
2026-01     7490
2026-02     8550
2026-03    11177
2026-04    12031
2026-05    12024
Freq: M, Name: count, dtype: int64


In [67]:
train = df[df['CloseMonth'] < TEST_MONTH].copy()
test  = df[df['CloseMonth'] == TEST_MONTH].copy()

In [73]:
train.shape, test.shape, len(test)/len(df)

((387133, 49), (12024, 49), 0.030123485245154163)

### Key Notes:
- At the moment, the X amount of months for the training set is 96% of the data, and while the exact percentage the train should make up should be determined through experimentation, I will aim to first make a 80-20 split to start
- I belive Normalization, Imputation, and Encoding (only True and False now) can be put through a pipeline in the coming week(s) when we begin actually training models, and conduct further feature engineering